In [ ]:
import numpy as np
import numpy.random as rnd
import numpy.linalg as linalg
import math
from scipy.special import softplus
from scipy.integrate import quad, quad_vec, cubature
import sys

Necessary functions for the AMP loop functions

In [ ]:
def QuasiNormal2D(X, mu, V):
    return(np.exp(-(np.einsum("ij,ij->i", (X - mu)@linalg.inv(V), (X - mu))/ 2)))

def Vtilde(V, z1):
    return(np.array(V) + np.array([[softplus(z1), 0], [0, 0]]))

def Pout(y, z0, z1):
    return(np.exp(-(y-z0)*(y-z0)/(2*softplus(z1)))/(np.sqrt(2*np.pi)*softplus(z1)))

def ZoutIntegrand(Z, Y, omega, V):
    return(Pout(Y, Z[:,0], Z[:,1])*QuasiNormal2D(Z, omega, V))

def dZoutIntegrand(Z, Y, omega, V):
    return(np.transpose((linalg.inv(V) @ np.transpose((Z - omega)))*Pout(Y, Z[:,0], Z[:,1])*QuasiNormal2D(Z, omega, V)))

def ddZoutIntegrand(Z, Y, omega, V):
    return(np.einsum("ijk,i->ijk",(np.einsum("ij,ik->ijk", np.transpose(linalg.inv(V) @ np.transpose((Z - omega))), np.transpose(linalg.inv(V) @ np.transpose((Z - omega)))) - linalg.inv(V)), Pout(Y, Z[:,0], Z[:,1])*QuasiNormal2D(Z, omega, V)))

AMP loop functions

In [ ]:
def InputChannel_BO(R, Sigma, SigmaInv):
    v = linalg.inv(SigmaInv + np.eye(2))
    a = v @ SigmaInv @ R
    return a, v

def GOutput_BO(omega, Y, V, IntLimits):
    Zout = cubature(f = ZoutIntegrand, a = np.array([-IntLimits, -IntLimits]), b = np.array([IntLimits, IntLimits]), args = (Y, omega, V)).estimate
    dZout = cubature(f = dZoutIntegrand, a = np.array([-IntLimits, -IntLimits]), b = np.array([IntLimits, IntLimits]), args = (Y, omega, V)).estimate
    ddZout = cubature(f = ddZoutIntegrand, a = np.array([-IntLimits, -IntLimits]), b = np.array([IntLimits, IntLimits]), args = (Y, omega, V)).estimate
    
    # Testing and comparing with numerical derrivative of gout : didn't coincide
    #omega0 = omega + np.array([1e-4, 0])
    #Zout0 = quad(lambda z1: Normal2D(np.array([Y, z1]), omega0, Vtilde(V, z1)), -IntLimits, IntLimits)[0]
    #dZout0 = quad_vec(lambda z1: linalg.inv(Vtilde(V, z1)) @ (np.array([Y, z1]) - omega0) * Normal2D(np.array([Y, z1]), omega0, Vtilde(V, z1)), -IntLimits, IntLimits)[0]
    #omega1 = omega + np.array([0, 1e-4])
    #Zout1 = quad(lambda z1: Normal2D(np.array([Y, z1]), omega1, Vtilde(V, z1)), -IntLimits, IntLimits)[0]
    #dZout1 = quad_vec(lambda z1: linalg.inv(Vtilde(V, z1)) @ (np.array([Y, z1]) - omega1) * Normal2D(np.array([Y, z1]), omega1, Vtilde(V, z1)), -IntLimits, IntLimits)[0]
    #gout = dZout/Zout
    #gout0 = dZout0/Zout0
    #gout1 = dZout1/Zout1
    #dGout_finite = np.array([[(gout[0] - gout0[0])/(1e-4), (gout[0] - gout1[0])/(1e-4)], [(gout[1] - gout0[1])/(1e-4), (gout[1] - gout1[1])/(1e-4)]])
    #print(omega)
    #print(V)
    #print(Zout)
    
    return (dZout/Zout),(ddZout/Zout - (dZout @ np.transpose(dZout))/(Zout*Zout)) #, dGout_finite #

def OutputChannel_BO(g, Dg, a, Xi, X2i):
    SigmaInv = -np.einsum("kjl,k->jl", Dg, X2i)
    Sigma = linalg.inv(SigmaInv)
    R = a + Sigma @ np.einsum("ki,k->i", g, Xi)
    return R, Sigma, SigmaInv

AMP runner function

In [ ]:
def GAMP_BO(X, Y, MaxIter = 1e4, EpsConvergence = 1e-6, Verbose = False, VerboseRate = 100, IntLimits = 20):
    X2 = X*X
    M = X.shape[0]
    d = X.shape[1]

    # Initializing variables
    a = np.zeros((d, 2))
    v = np.tile(np.eye(2), (d, 1, 1))
    Sigma = np.tile(np.eye(2), (d, 1, 1))
    SigmaInv = np.tile(np.eye(2), (d, 1, 1))
    g = np.zeros((M, 2))
    NIter = 0
    Conv = 1

    while((Conv > EpsConvergence) and (NIter < MaxIter)):
        # Updating mean and variance
        V = np.einsum("ki,ijl->kjl", X2, v)
        omega = np.einsum("il,ki->kl", a, X) - np.einsum("ki,ijl,ilm,ima,ka->kj", X2, SigmaInv, v, Sigma, g)
        newg, newDg = zip(*[GOutput_BO(omega[k,:], Y[k], V[k,:,:], IntLimits) for k in range(len(omega))])
        # Updating output channel
        g = np.array(newg)
        Dg = np.array(newDg)
        newR, newSigma, newSigmaInv = zip(*[OutputChannel_BO(g, Dg, a[i,:], X[:,i], X2[:,i]) for i in range(len(a))])
        R = np.array(newR)
        Sigma = np.array(newSigma)
        SigmaInv = np.array(newSigmaInv)
        # Updating input channel
        newa, newv = zip(*[InputChannel_BO(R[i,:], Sigma[i,:,:], SigmaInv[i,:,:]) for i in range(len(R))])
        Conv = np.sum(np.abs(a - np.array(newa)))
        a = np.array(newa)
        v = np.array(newv)
        if(Verbose and NIter%VerboseRate == 0):
            print("Iteration %s" % NIter)
            print("Current convergence criterion %s" % Conv)
        NIter += 1
    return a, v

Main

In [ ]:
d = 100
M = 200
X = rnd.normal(0, 1/np.sqrt(d), size = (M, d))
wvTrue = rnd.normal(0, 1, size = (d, 2))
yTrue = np.matmul(X, wvTrue[:,0])
yNoisy = yTrue + rnd.normal(0, np.sqrt(softplus(np.matmul(X, wvTrue[:,1]))))
a, v = GAMP_BO(X, yNoisy, MaxIter = 100, EpsConvergence = 1e-6, Verbose = True, VerboseRate = 1, IntLimits = 5)

Iteration 0
Current convergence criterion 199.47976243999346
Iteration 1
Current convergence criterion 387.06720762489994
Iteration 2
Current convergence criterion 206.3558839056525


C:\Users\Adam\AppData\Local\Temp\ipykernel_19924\3822694000.py:26: RuntimeWarning: overflow encountered in matmul
  return (dZout/Zout),(ddZout/Zout - (dZout @ np.transpose(dZout))/(Zout*Zout)) #, dGout_finite #
C:\Users\Adam\AppData\Local\Temp\ipykernel_19924\3822694000.py:26: RuntimeWarning: overflow encountered in double_scalars
  return (dZout/Zout),(ddZout/Zout - (dZout @ np.transpose(dZout))/(Zout*Zout)) #, dGout_finite #
C:\Users\Adam\AppData\Local\Temp\ipykernel_19924\3822694000.py:26: RuntimeWarning: invalid value encountered in double_scalars
  return (dZout/Zout),(ddZout/Zout - (dZout @ np.transpose(dZout))/(Zout*Zout)) #, dGout_finite #


Iteration 3
Current convergence criterion nan
